In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [3]:
from huggingface_hub import snapshot_download

repos = [
    'deepghs/arknights_voices_kr',
    'deepghs/arknights_voices_jp',
    'deepghs/arknights_voices_zh',
    'deepghs/arknights_voices_en',
]

for r in repos:
    snapshot_download(
        repo_id=r, 
        repo_type="dataset", local_dir=r.split('/')[1])

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:04<00:00,  1.00it/s]


In [5]:
files = glob('arknights*/*.tar')
files

['arknights_voices_jp/voices.tar',
 'arknights_voices_en/voices.tar',
 'arknights_voices_kr/voices.tar',
 'arknights_voices_zh/voices.tar']

In [7]:
def loop(files):
    files, _ = files
    import tarfile

    for f in tqdm(files):
        with tarfile.open(f, "r") as tar:
            tar.extractall(path=f.split('/')[0])
        os.remove(f)

In [8]:
multiprocessing(files, loop, cores = len(files), returned = False)

100%|██████████| 1/1 [00:04<00:00,  4.81s/it]


In [27]:
files = glob('*/table.parquet')
files

['arknights_voices_jp/table.parquet',
 'arknights_voices_en/table.parquet',
 'arknights_voices_kr/table.parquet',
 'arknights_voices_zh/table.parquet']

In [28]:
rows = []
for f in files:
    print(f)
    df = pd.read_parquet(f)
    f_ = f.split('/')[0]
    for i in range(len(df)):
        t = df['voice_text'].iloc[i].strip()
        if len(t) < 2:
            continue
        id = df['id'].iloc[i]
        f = f'{f_}/{id}.wav'
        if not os.path.exists(f):
            continue
        rows.append({
            'audio_filename': f,
            'text': t,
            'speaker': f"{f_}_{df['voice_actor_name'].iloc[i]}"
        })
len(rows)

arknights_voices_jp/table.parquet
arknights_voices_en/table.parquet
arknights_voices_kr/table.parquet
arknights_voices_zh/table.parquet


41548

In [31]:
rows[-2]['audio_filename']

{'audio_filename': 'arknights_voices_zh/char_431_ashlok_CN_042.wav',
 'text': '今天有什么任务？',
 'speaker': 'arknights_voices_zh_龟娘'}

In [36]:
import copy

def loop(rows):
    rows, _ = rows
    data = []
    for r in tqdm(rows):
        r = copy.copy(r)
        base = r['audio_filename'].split('/')[0] + '_audio'
        audio_filename = r['audio_filename'].replace('/', '-').replace('.wav', '.mp3')
        os.makedirs(base, exist_ok=True)
        audio_filename = os.path.join(base, audio_filename)
        audio_np, sr = sf.read(r['audio_filename'])
        if audio_np.ndim > 1:
            audio_np = audio_np.mean(axis=1)
        if audio_np.shape[0] < 10000:
            continue
        sf.write(audio_filename, audio_np, sr)
        r['audio_filename'] = audio_filename
        data.append(r)
    return data

In [37]:
data = loop((rows[:10], 0))

100%|██████████| 10/10 [00:02<00:00,  3.70it/s]


In [39]:
data = multiprocessing(rows, loop, cores = 30)

100%|██████████| 1384/1384 [02:55<00:00,  7.87it/s]


In [41]:
with open('arknights_voices.json', 'w') as fopen:
    json.dump(data, fopen)

In [42]:
audio_files = [d['audio_filename'] for d in data]

with open('arknights_voices-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [46]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'arknights_voices_jp_audio/arknights_voices_jp-char_427_vigil_CN_001.mp3',
 'text': '書類の整理？ドクター、それよりもっと大事な仕事があるはずだ。……いいだろう、そこまで言うなら手を貸す。だが、気乗りしない仕事に見合うだけのプランと引き換えだ。失望させてくれるなよ。',
 'speaker': 'arknights_voices_jp_KENN'}

In [47]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'arknights_voices')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 48.85ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  86%|████████▌ | 2.80MB / 3.27MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 3.27MB / 3.27MB, 2.37MB/s  
Processing Files (1 / 1): 100%|██████████| 3.27MB / 3.27MB,  591kB/s  
New Data Upload: 100%|██████████| 3.27MB / 3.27MB,  591kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.17s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/ffe8781a327fda9ab7b7ca7af288d19874c2ce45', commit_message='Upload dataset', commit_description='', oid='ffe8781a327fda9ab7b7ca7af288d19874c2ce45', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [50]:
# for f in glob('arknights*_audio'):
#     print(f)
#     os.system(f'zip -rq {f}.zip {f}')

In [51]:
# for f in glob('arknights*_audio.zip'):
#     print(f)
#     os.system(f'hf upload malaysia-ai/Multilingual-TTS {f} --repo-type=dataset')